In [4]:
#import kagglehub
import os
import pandas as pd

In [ ]:
# Download latest version
#path = kagglehub.dataset_download("wcukierski/enron-email-dataset")

#print("Path to dataset files:", path)

#PATH: C:\Users\mikec\.cache\kagglehub\datasets\wcukierski\enron-email-dataset\versions\2


In [6]:
path = 'C:\\Users\\mikec\\.cache\\kagglehub\\datasets\\wcukierski\\enron-email-dataset\\versions\\2'

In [15]:
csv_path = os.path.join(path, "emails.csv")  # adjust name if needed
df = pd.read_csv(csv_path)

In [16]:

df.shape

(517401, 2)

In [17]:
df.columns

Index(['file', 'message'], dtype='object')

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   file     517401 non-null  object
 1   message  517401 non-null  object
dtypes: object(2)
memory usage: 7.9+ MB


In [21]:
df.head(10)

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...
5,allen-p/_sent_mail/1002.,Message-ID: <30965995.1075863688265.JavaMail.e...
6,allen-p/_sent_mail/1003.,Message-ID: <16254169.1075863688286.JavaMail.e...
7,allen-p/_sent_mail/1004.,Message-ID: <17189699.1075863688308.JavaMail.e...
8,allen-p/_sent_mail/101.,Message-ID: <20641191.1075855687472.JavaMail.e...
9,allen-p/_sent_mail/102.,Message-ID: <30795301.1075855687494.JavaMail.e...


In [24]:
from email import policy
from email.parser import Parser
import re

HEADER_RE = re.compile(r"^([\w\-]+):\s*(.*)$")

def parse_email_robust(raw):
    if not isinstance(raw, str):
        return {
            "message_id": None, "from": None, "to": None, "cc": None, "bcc": None,
            "date": None, "subject": None, "body": None
        }

    # Split headers/body at first blank line
    parts = raw.split("\n\n", 1)
    header_text = parts[0]
    body = parts[1] if len(parts) > 1 else ""

    # Handle folded headers (lines starting with whitespace continue previous header)
    lines = header_text.splitlines()
    unfolded = []
    for line in lines:
        if line.startswith((" ", "\t")) and unfolded:
            unfolded[-1] += " " + line.strip()
        else:
            unfolded.append(line.strip())

    headers = {}
    for line in unfolded:
        m = HEADER_RE.match(line)
        if m:
            k, v = m.group(1).lower(), m.group(2).strip()
            # If header repeats, keep all values
            headers[k] = (headers[k] + ", " + v) if k in headers else v

    return {
        "message_id": headers.get("message-id"),
        "from": headers.get("from"),
        "to": headers.get("to"),
        "cc": headers.get("cc"),
        "bcc": headers.get("bcc"),
        "date": headers.get("date"),
        "subject": headers.get("subject"),
        "body": body.strip() or None
    }



In [25]:
parsed_df = df["message"].map(parse_email_robust).apply(pd.Series)

df_clean = pd.concat([df.drop(columns=["message"]), parsed_df], axis=1)

df_clean.head()


,file,message_id,from,to,cc,bcc,date,subject,body
0,allen-p/_sent_mail/1.,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,tim.belden@enron.com,None,None,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",,Here is our forecast
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,john.lavorato@enron.com,None,None,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",Re:,Traveling to have a business meeting takes the...
2,allen-p/_sent_mail/100.,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,leah.arsdall@enron.com,None,None,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",Re: test,test successful. way to go!!!
3,allen-p/_sent_mail/1000.,<13505866.1075863688222.JavaMail.evans@thyme>,phillip.allen@enron.com,randall.gay@enron.com,None,None,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",,"Randy,\n\n Can you send me a schedule of the s..."
4,allen-p/_sent_mail/1001.,<30922949.1075863688243.JavaMail.evans@thyme>,phillip.allen@enron.com,greg.piper@enron.com,None,None,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",Re: Hello,Let's shoot for Tuesday at 11:45.


In [26]:
df_clean["employee"] = df_clean["file"].str.split("/").str[0]
df_clean["folder"]   = df_clean["file"].str.split("/").str[1]


In [27]:
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["body_length"] = df_clean["body"].fillna("").str.len()
df_clean["subject_length"] = df_clean["subject"].fillna("").str.len()


In [28]:
df_clean.isna().sum()


file                   0
message_id             0
from                   0
to                 21847
cc                389520
bcc               389520
date                   1
subject                0
body                   0
employee               0
folder                 0
body_length            0
subject_length         0
dtype: int64

In [29]:
# Fill missing subject with placeholder
df_clean["subject"] = df_clean["subject"].fillna("(no subject)")

# Drop rows with no date (optional, depends on analysis)
df_clean = df_clean.dropna(subset=["date"])

In [30]:
df_clean.isna().sum()

file                   0
message_id             0
from                   0
to                 21847
cc                389519
bcc               389519
date                   0
subject                0
body                   0
employee               0
folder                 0
body_length            0
subject_length         0
dtype: int64

In [32]:
random_state=85288

df_sample = df_clean.sample(n=50, random_state=random_state)

In [33]:
df_sample.head(5)

,file,message_id,from,to,cc,bcc,date,subject,body,employee,folder,body_length,subject_length
145734,guzman-m/_sent_mail/295.,<6593456.1075840743847.JavaMail.evans@thyme>,mark.guzman@enron.com,jubran.whalan@enron.com,None,None,2000-09-26 00:09:00-07:00,Re: Sleeves,I will make sure these are input today. Thank...,guzman-m,_sent_mail,86,11
320454,mann-k/sent/32.,<13574566.1075845919284.JavaMail.evans@thyme>,kay.mann@enron.com,j.nassos@pecorp.com,"dtauke@winston.com, frederik_aldin@westlb.com,...","dtauke@winston.com, frederik_aldin@westlb.com,...",2000-06-06 10:25:00-07:00,Re: ABB Transformer Option Agreement,"My only comment is that the word ""lease"" shoul...",mann-k,sent,1336,36
87541,dean-c/inbox/747.,<2229044.1075852155435.JavaMail.evans@thyme>,40enron@enron.com,None,None,None,2001-10-05 00:17:09-07:00,Important Information About Security & ClickAt...,"Dear ClickAtHome Participant,\n\nClickAtHome i...",dean-c,inbox,1963,50
18945,baughman-d/discussion_threads/47.,<8520205.1075848322377.JavaMail.evans@thyme>,rhonda.denton@enron.com,"tim.belden@enron.com, dana.davis@enron.com, ge...",None,None,2000-12-20 08:25:00-08:00,Split Rock Energy LLC,We have received the executed EEI contract fro...,baughman-d,discussion_threads,123,21
347972,nemec-g/all_documents/6451.,<21099108.1075842802558.JavaMail.evans@thyme>,ronr@hba.org,ronr@hba.org,None,None,2001-06-07 06:37:00-07:00,HBA Pictorial Roster Photos - FREE,Photo Sessions for Next Pictorial Roster Set f...,nemec-g,all_documents,1124,34


In [35]:

df_clean.shape

(517400, 13)

In [34]:
# =========================
# USER CONFIGURATION
# =========================
export_dir = r"C:\Users\mikec\OneDrive\Desktop\Doc School\Dissertation Build\datasets"   # 👈 CHANGE THIS
sample_size = 1000
random_seed = 85288

# =========================
# CREATE DIRECTORY IF NEEDED
# =========================
os.makedirs(export_dir, exist_ok=True)

# =========================
# FILE PATHS
# =========================
full_path = os.path.join(export_dir, "enron_df_clean_full.csv")
sample_path = os.path.join(export_dir, "enron_df_clean_sample_1000.csv")

# =========================
# EXPORT FULL DATASET
# =========================
df_clean.to_csv(full_path, index=False)

# =========================
# CREATE & EXPORT SAMPLE
# =========================
df_sample = df_clean.sample(
    n=min(sample_size, len(df_clean)),
    random_state=random_seed
)

df_sample.to_csv(sample_path, index=False)

# =========================
# CONFIRMATION
# =========================
print(f"Full dataset saved to: {full_path}")
print(f"Sample dataset saved to: {sample_path}")
print(f"Full size: {len(df_clean):,} rows")
print(f"Sample size: {len(df_sample):,} rows")


Full dataset saved to: C:\Users\mikec\OneDrive\Desktop\Doc School\Dissertation Build\datasets\enron_df_clean_full.csv
Sample dataset saved to: C:\Users\mikec\OneDrive\Desktop\Doc School\Dissertation Build\datasets\enron_df_clean_sample_1000.csv
Full size: 517,400 rows
Sample size: 1,000 rows
